In [7]:
# Umbilical Cord Blood Transplantation (UCBT) Dataset Creation
# Purpose: Define the UCBT dataset schema and create an structure to store patient data from multiple studies,
# enabling incremental addition of study data with robust parsing & realistic dose sampling.

import os, re, sys, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import ks_2samp, chisquare, lognorm

print(f"Python executable: {sys.executable}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Base path & output
BASE = Path("/Users/amanda/Desktop/UCBT")
OUT = BASE / "ucbt_dataset.csv"
BASE.mkdir(parents=True, exist_ok=True)


# Schema & dtypes
columns = [
    'Patient_ID',
    'Recipient_Age',
    'Recipient_Sex',         # 0=Male, 1=Female (Int64 later)
    'Ethnicity',
    'Race',
    'Disease_Type',
    'Conditioning_Regimen',
    'Remission_Status',      # ordinal: 0=Active/Relapse, 1=CR1, 2=CR2, 3=CR3+
    'Cord_Blood_Units',      # 0=Single, 1=Double 
    'HLA_Match_Level',
    'CD34_Dose',             # string like 'a x 10^5/kg' or bins, or 'Missing/Unknown'
    'TNC_Dose',              # string like 'a x 10^7/kg' or bins, or 'Missing/Unknown'
    'Neutrophil_Engraftment',# Int64, 0/1
    'Platelet_Engraftment',  # Int64, 0/1
    'Chronic_GVHD',          # Int64, 0/1
    '1_Year_Survival',       # Int64, 0/1
    'Study_ID'
]

dtypes = {
    'Patient_ID': 'int64',
    'Recipient_Age': 'float64',
    'Recipient_Sex': 'float64',
    'Ethnicity': 'object',
    'Race': 'object',
    'Disease_Type': 'object',
    'Conditioning_Regimen': 'object',
    'Remission_Status': 'float64',
    'Cord_Blood_Units': 'float64',
    'HLA_Match_Level': 'object',
    'CD34_Dose': 'object',
    'TNC_Dose': 'object',
    'Neutrophil_Engraftment': 'float64',
    'Platelet_Engraftment': 'float64',
    'Chronic_GVHD': 'float64',
    '1_Year_Survival': 'float64',
    'Study_ID': 'object',
}

dataset = pd.DataFrame(columns=columns).astype(dtypes)


# Helpers: parsing, sampling, HLA, etc.
_num_re = re.compile(r'(?P<a>\d+(?:\.\d+)?)')
_sci_re = re.compile(r'(?P<base>\d+(?:\.\d+)?)\s*[x×]\s*10(?:\^|\\\^)?(?P<exp>[-+]?\d+)', re.I)

def _midpoint_from_range_txt(txt: str):
    if not isinstance(txt, str): return np.nan
    part = txt.split('x10')[0] if 'x10' in txt else txt
    nums = _num_re.findall(part)
    vals = [float(n) for n in nums]
    return np.mean(vals[:2]) if len(vals) >= 2 else (vals[0] if vals else np.nan)

def _sample_from_inequality(s, target_exp, jitter=0.05):
    m = _sci_re.search(s)
    if m:
        thr = float(m.group('base')) * (10 ** (int(m.group('exp')) - target_exp))
    else:
        n = _num_re.search(s)
        thr = float(n.group('a')) if n else np.nan
    if np.isnan(thr):
        return np.nan
    if s.startswith('<=') or s.startswith('<'):
        lo, hi = max(thr*(1-jitter), 0.0), thr
    else:  # >= or >
        lo, hi = thr, thr*(1+jitter)
    if hi - lo < 1e-9:  # widen if degenerate
        hi = thr*1.10 if thr > 0 else 0.1
    return float(np.random.uniform(lo, hi))

def _as_scaled_number(txt, target_exp: int):
    """Return float scaled to 10^target_exp for 'a x 10^b' formats, ranges & inequalities with realistic sampling."""
    if txt is None or (isinstance(txt, float) and np.isnan(txt)): return np.nan
    if isinstance(txt, (int, float)): return float(txt)
    s = str(txt).replace('×','x').replace('≥','>=').replace('≤','<=').strip()
    # scientific range (e.g. '3-5x10^7/kg')
    if '-' in s and 'x10' in s:
        mid = _midpoint_from_range_txt(s)
        m = _sci_re.search(s)
        return mid * (10 ** (int(m.group('exp')) - target_exp)) if m and not np.isnan(mid) else mid
    # inequalities: sample in a band near the threshold
    if any(s.startswith(op) for op in ('>=','<=','>','<')):
        return _sample_from_inequality(s, target_exp)
    # plain scientific
    m = _sci_re.search(s)
    if m: return float(m.group('base')) * (10 ** (int(m.group('exp')) - target_exp))
    # plain numeric
    n = _num_re.search(s)
    return float(n.group('a')) if n else np.nan

def parse_cd34(series: pd.Series):
    """CD34+ numeric to 10^5/kg."""
    return series.apply(lambda v: _as_scaled_number(v, 5))

def parse_tnc(series: pd.Series):
    """TNC numeric to 10^7/kg."""
    return series.apply(lambda v: _as_scaled_number(v, 7))

def remission_ordinal(val):
    if pd.isna(val): return np.nan
    s = str(val).upper()
    if any(k in s for k in ['RELAP','ACTIVE']): return 0
    if 'CR1' in s or s=='1': return 1
    if 'CR2' in s or s=='2': return 2
    if 'CR3' in s or 'CR3+' in s or s=='3': return 3
    try: return float(val)
    except: return np.nan

def _split_multi_labels(s: str):
    """Split HLA tokens by ',', '–', '-', ' or '."""
    if s is None: return []
    s = str(s)
    # normalise ' or ' to comma
    s = s.replace(' or ', ',').replace(' OR ', ',')
    # split by comma first
    parts = []
    for chunk in s.split(','):
        parts.extend(re.split(r'[–-]', chunk))
    return [p.strip() for p in parts if p.strip()]

def hla_to_features(hla):
    """From '4/6', '5/6,6/6', '4–6/6', '<=5/8', etc. -> hla_best_pre, hla_worst_pre, hla_any6_pre, hla_double_pre."""
    if pd.isna(hla):
        return pd.Series({'hla_best_pre': np.nan,'hla_worst_pre': np.nan,'hla_any6_pre': np.nan,'hla_double_pre': np.nan})
    parts = _split_multi_labels(hla)
    vals = []
    for p in parts:
        # range like 4-6/6 already split: capture each fraction
        # inequalities like '<=5/8'
        m_ineq = re.match(r'(<=|>=|<|>)(\d+)/(\d+)', p)
        if m_ineq:
            num = int(m_ineq.group(2)); den = int(m_ineq.group(3))
            if den: vals.append(num/den)
            continue
        m = re.search(r'(\d+)/(\d+)', p)
        if m:
            num = int(m.group(1)); den = int(m.group(2))
            if den: vals.append(num/den)
    if not vals:
        return pd.Series({'hla_best_pre': np.nan,
                          'hla_worst_pre': np.nan,
                          'hla_any6_pre': np.nan,
                          'hla_double_pre': float(len(parts) > 1)})
    return pd.Series({
        'hla_best_pre':  np.nanmax(vals),
        'hla_worst_pre': np.nanmin(vals),
        'hla_any6_pre':  float(any(abs(v - 1.0) < 1e-9 for v in vals)),
        'hla_double_pre': float(len(parts) > 1)
    })

def _emit_dose_string(val, exp):
    """Return standardized 'a x 10^exp/kg' string from numeric; pass through strings."""
    try:
        v = float(val)
    except:
        return val
    return f"{v}x10^{exp}/kg"

def generate_field_values(n, field_data, field_name="Field"):
    # direct list
    if isinstance(field_data, list):
        if len(field_data) != n:
            raise ValueError(f"{field_name}: List length {len(field_data)} != n={n}")
        return field_data
    # tuple: (edges, probs)
    if isinstance(field_data, tuple) and len(field_data)==2:
        edges, probs = field_data
        probs = np.array(probs, float); probs = probs / probs.sum()
        out = np.empty(n)
        for i in range(n):
            j = np.random.choice(len(edges)-1, p=probs)
            out[i] = np.random.uniform(edges[j], edges[j+1])
        return out
    # dict with 'range'
    if isinstance(field_data, dict) and set(field_data.keys())=={"range"}:
        lo, hi = field_data["range"]
        return np.random.uniform(lo, hi, size=n)
    # dict with 'median' and 'range'
    if isinstance(field_data, dict) and {"median","range"}.issubset(field_data.keys()):
        med = float(field_data["median"]); lo, hi = field_data["range"]
        sd = (hi-lo)/6 if hi>lo else 1.0
        vals = np.random.normal(med, sd, n)
        return np.clip(vals, lo, hi)
    # dict with 'bins' & 'counts'
    if isinstance(field_data, dict) and {"bins","counts"}.issubset(field_data.keys()):
        bins, counts = field_data["bins"], field_data["counts"]
        w = np.array(list(counts), dtype=float)
        probs = w / w.sum() if w.sum() > 0 else np.ones_like(w)/len(w)
        out = np.empty(n)
        for i in range(n):
            j = np.random.choice(len(bins), p=probs)
            lo, hi = bins[j]
            out[i] = np.random.uniform(lo, hi)
        return out
    # flat dict of labels -> {prob or count}
    if isinstance(field_data, dict):
        keys = list(field_data.keys())
        w = np.array(list(field_data.values()), dtype=float)
        if (w > 1.0+1e-9).any():
            probs = w / max(w.sum(), 1.0)
        else:
            probs = w / w.sum()
        return np.random.choice(keys, size=n, p=probs)
    # scalar prob for outcomes
    if isinstance(field_data, (int,float)) and field_name in [
        'Neutrophil_Engraftment','Platelet_Engraftment','Chronic_GVHD','1_Year_Survival'
    ]:
        p = float(field_data); p = 0.0 if p<0 else (1.0 if p>1 else p)
        return (np.random.rand(n) < p).astype(int)
    raise ValueError(f"{field_name}: Unrecognised format {type(field_data)}")

def _materialize_dose_field(n, spec, default_exp, study_non_missing=None, global_median=None, use_synthetic=True):
    if spec is None:
        return None
    # Numeric distribution spec
    if isinstance(spec, dict) and {"median","range"}.issubset(spec.keys()):
        exp = int(spec.get("unit_exp", default_exp))
        vals = generate_field_values(n, {"median": float(spec["median"]), "range": tuple(spec["range"])}, "dose")
        return [_emit_dose_string(v, exp) for v in vals]

    # Categorical/binned labels or lists
    out = generate_field_values(n, spec, "dose")
    # Wrap any pure numerics with default_exp, leave existing strings untouched
    arr = [v if isinstance(v, str) and ('x10' in v or '×10' in v) else _emit_dose_string(v, default_exp) for v in out]

    # Impute 'Missing/Unknown'
    arr = np.array(arr, dtype=object)
    missing_mask = (arr == 'Missing/Unknown')
    if missing_mask.any():
        use_pool = None
        # sanitise study pool
        if study_non_missing is not None:
            study_non_missing = np.asarray(study_non_missing, dtype=float)
            study_non_missing = study_non_missing[~np.isnan(study_non_missing) & (study_non_missing > 0)]
            if len(np.unique(np.round(study_non_missing, 4))) >= 20:
                use_pool = study_non_missing

        if use_synthetic and use_pool is not None and len(use_pool) > 10:
            shape, loc, scale = lognorm.fit(use_pool, floc=0)
            imputed_vals = lognorm.rvs(shape, loc, scale, size=missing_mask.sum(), random_state=RANDOM_STATE)
            # Soft constraints for plausibility
            lo_clip = np.percentile(use_pool, 2)
            hi_clip = np.percentile(use_pool, 98)
            imputed_vals = np.clip(imputed_vals, max(lo_clip, 0.01), hi_clip)
        elif use_pool is not None and len(use_pool) > 0:
            imputed_vals = np.random.choice(use_pool, size=missing_mask.sum())
        else:
            # fallback to a single-value "median"
            fallback = global_median if (global_median is not None and not np.isnan(global_median)) else 1.0
            imputed_vals = np.full(missing_mask.sum(), fallback)

        arr[missing_mask] = [_emit_dose_string(iv, default_exp) for iv in imputed_vals]
    return list(arr)


# Ingestion
def _ensure_unique_ids(df_all: pd.DataFrame, id_col='Patient_ID'):
    """Regenerate IDs for duplicates to guarantee global uniqueness."""
    # Unlikely, but keep looping until all unique
    while df_all[id_col].duplicated().any():
        mask = df_all[id_col].duplicated()
        df_all.loc[mask, id_col] = np.random.randint(100000, 999999, size=mask.sum())
    return df_all

def add_study_data(study_data_dict: dict, study_id: str) -> pd.DataFrame:
    global dataset
    n = int(study_data_dict.get('n_patients', 0))
    if n <= 0:
        raise ValueError(f"{study_id}: n_patients must be >0")
    df = pd.DataFrame(index=range(n), columns=columns)
    # unique IDs
    df['Patient_ID'] = np.random.randint(100000, 999999, size=n)
    df['Study_ID'] = study_id

    # Missingness helper
    def fill_or_unknown(col, sentinel="Unknown"):
        data = study_data_dict.get(col, None)
        missing_flag = 1 if data is None else 0
        if data is None:
            df[col] = sentinel
        else:
            df[col] = generate_field_values(n, data, field_name=col) if not isinstance(data, (int,float,str)) else (
                [data]*n if col not in ['Recipient_Age'] else generate_field_values(n, data, field_name=col)
            )
        return missing_flag

    # Text cats that can be "Unknown"
    df['Ethnicity_missing'] = fill_or_unknown('Ethnicity', 'Unknown')
    df['Race_missing'] = fill_or_unknown('Race', 'Unknown')
    df['Conditioning_Regimen_missing'] = fill_or_unknown('Conditioning_Regimen', 'Unknown')

    # Harmonise regimen labels
    s = df['Conditioning_Regimen'].astype(str).str.strip().str.lower()
    mp = {
        'myeloablative': 'Myeloablative',
        'myeloablative conditioning': 'Myeloablative',
        'mac': 'Myeloablative',
        'reduced intensity': 'Reduced intensity',
        'reduced-intensity': 'Reduced intensity',
        'reduced intensity conditioning': 'Reduced intensity',
        'ric': 'Reduced intensity',
        'other': 'Other',
        'unknown': 'Unknown',
        'missing': 'Unknown'
    }
    df['Conditioning_Regimen'] = s.map(mp).fillna(df['Conditioning_Regimen'])

    # Age
    age = study_data_dict.get('Recipient_Age', None)
    if age is not None:
        df['Recipient_Age'] = generate_field_values(n, age, 'Recipient_Age')

    # Sex (0/1) — allow NaN if missing
    sex = study_data_dict.get('Recipient_Sex', None)
    df['Recipient_Sex_missing'] = 1 if sex is None else 0
    if sex is not None:
        vals = generate_field_values(n, sex, 'Recipient_Sex')
        mp_sex = {'M':0, 'F':1, 'Male':0, 'Female':1, '0':0, '1':1}
        vals = pd.Series(vals).map(mp_sex).fillna(pd.to_numeric(pd.Series(vals), errors='coerce'))
        df['Recipient_Sex'] = vals.values

    # Disease (coarse map)
    df['Disease_Type'] = generate_field_values(n, study_data_dict.get('Disease_Type','Unknown'), 'Disease_Type')
    _dt = pd.Series(df['Disease_Type'], dtype="object").astype(str).str.lower()
    coarse = []
    for x in _dt:
        if 'aml' in x: coarse.append('AML')
        elif 'all' in x: coarse.append('ALL')
        elif 'cml' in x: coarse.append('CML')
        elif 'mds' in x or 'myelodysplastic' in x: coarse.append('MDS')
        else: coarse.append('Other')
    df['Disease_Type'] = coarse

    # Remission (ordinal)
    rem = study_data_dict.get('Remission_Status', None)
    df['Remission_Status_missing'] = 1 if rem is None else 0
    if rem is not None:
        rvals = generate_field_values(n, rem, 'Remission_Status')
        df['Remission_Status'] = pd.Series(rvals).apply(remission_ordinal).values

    # Cord units (0 single / 1 double)
    cbu = study_data_dict.get('Cord_Blood_Units', None)
    if cbu is None:
        df['Cord_Blood_Units'] = np.nan
    else:
        vals = generate_field_values(n, cbu, 'Cord_Blood_Units')
        mp_cbu = {'Single':0, 'Double':1, '0':0, '1':1}
        df['Cord_Blood_Units'] = pd.Series(vals).map(mp_cbu).fillna(pd.to_numeric(pd.Series(vals), errors='coerce')).values

    # HLA level (store raw, plus engineered features)
    df['HLA_Match_Level'] = generate_field_values(n, study_data_dict.get('HLA_Match_Level','Unknown'), 'HLA_Match_Level')

    # CD34 / TNC with unit support and imputation
    # global medians from already parsed prior rows (numeric pools), else sensible fallback
    global_cd34_median = dataset.get('CD34_num', pd.Series(dtype=float)).median() if 'CD34_num' in dataset.columns else 2.0
    global_tnc_median  = dataset.get('TNC_num',  pd.Series(dtype=float)).median() if 'TNC_num'  in dataset.columns else 3.0

    # First pass: create strings according to study spec (to estimate study pool)
    cd34_spec = study_data_dict.get('CD34_Dose', None)
    if cd34_spec is None:
        df['CD34_Dose'] = 'Missing/Unknown'
    else:
        dose_strings_tmp = _materialize_dose_field(n, cd34_spec, default_exp=5)
        # Create study pool by parsing the temp strings (excluding Missing)
        non_missing_vals = [parse_cd34(pd.Series([d]))[0] for d in dose_strings_tmp if d != 'Missing/Unknown']
        df['CD34_Dose'] = _materialize_dose_field(n, cd34_spec, default_exp=5,
                                                  study_non_missing=non_missing_vals,
                                                  global_median=global_cd34_median,
                                                  use_synthetic=True)
    df['CD34_Dose_missing'] = 1 if cd34_spec is None else 0

    tnc_spec = study_data_dict.get('TNC_Dose', None)
    if tnc_spec is None:
        df['TNC_Dose'] = 'Missing/Unknown'
    else:
        dose_strings_tmp = _materialize_dose_field(n, tnc_spec, default_exp=7)
        non_missing_vals = [parse_tnc(pd.Series([d]))[0] for d in dose_strings_tmp if d != 'Missing/Unknown']
        df['TNC_Dose'] = _materialize_dose_field(n, tnc_spec, default_exp=7,
                                                 study_non_missing=non_missing_vals,
                                                 global_median=global_tnc_median,
                                                 use_synthetic=True)
    df['TNC_Dose_missing'] = 1 if tnc_spec is None else 0

    # Outcomes
    for y in ['Neutrophil_Engraftment','Platelet_Engraftment','Chronic_GVHD','1_Year_Survival']:
        val = study_data_dict.get(y, None)
        if val is None:
            df[y] = np.nan
        else:
            out = generate_field_values(n, val, y)
            df[y] = pd.to_numeric(pd.Series(out), errors='coerce')

    # Cleaning / casting
    df['Recipient_Age'] = pd.to_numeric(df['Recipient_Age'], errors='coerce').clip(0,115)

    # Early numeric dose (unit-scaled)
    df['CD34_num_pre'] = parse_cd34(df['CD34_Dose'])
    df['TNC_num_pre']  = parse_tnc(df['TNC_Dose'])

    # HLA features
    h = df['HLA_Match_Level'].apply(hla_to_features)
    for c in ['hla_best_pre','hla_worst_pre','hla_any6_pre','hla_double_pre']:
        if c in h.columns:
            df[c] = pd.to_numeric(h[c], errors='coerce')

    # Cast outcomes to nullable Int64, sex/cord to Int64 where possible
    for y in ['Neutrophil_Engraftment','Platelet_Engraftment','Chronic_GVHD','1_Year_Survival']:
        if y in df:
            df[y] = pd.to_numeric(df[y], errors='coerce').round().astype('Int64')
    if 'Recipient_Sex' in df:
        df['Recipient_Sex'] = pd.to_numeric(df['Recipient_Sex'], errors='coerce').round().astype('Int64')
    if 'Cord_Blood_Units' in df:
        df['Cord_Blood_Units'] = pd.to_numeric(df['Cord_Blood_Units'], errors='coerce').round().astype('Int64')

    # Stage -> final numeric dose columns (retain *_pre for debug; create final and keep both)
    df['CD34_num'] = df['CD34_num_pre']
    df['TNC_num']  = df['TNC_num_pre']

    # Append & enforce dtypes softly
    df = df.astype(dtypes, errors='ignore')
    dataset = pd.concat([dataset, df], ignore_index=True)

    # Ensure global unique Patient_IDs
    _ensure_unique_ids(dataset, 'Patient_ID')

    print(f"Added {n} patients from {study_id}")
    return dataset

def save_dataset(path: str, df: pd.DataFrame) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f"Dataset saved at: {path}")

def validate_dataset(df: pd.DataFrame) -> None:
    print("\nVALIDATION ")
    # Missingness snapshot
    miss = df.isna().mean().sort_values(ascending=False)
    print("Top missing columns:")
    print(miss.head(12))

    # Age range
    ok = df['Recipient_Age'].dropna().between(0,115)
    if (~ok).any():
        warnings.warn("Recipient_Age outside [0,115] detected.")

    # Study-level medians for numeric doses
    for col, exp in [('CD34_num', 5), ('TNC_num', 7)]:
        if col in df.columns and df[col].notna().any():
            g = df.groupby('Study_ID')[col].median().sort_values()
            print(f"\nMedian {col} by Study_ID (expected 10^{exp}/kg scale):")
            try:
                print(g.head(6)); print("..."); print(g.tail(6))
            except Exception:
                print(g)

    # AgeBin preview
    if 'Recipient_Age' in df.columns:
        tmp = pd.cut(pd.to_numeric(df['Recipient_Age'], errors='coerce'),
                     bins=[0,5,10,18,40,60,115], include_lowest=True).astype(str)
        print("\nAgeBin preview:", tmp.value_counts(dropna=False).head())


# Studies 

# Study 1
study1_data = {
    'n_patients': 885,
    'Recipient_Age': ([0, 9, 17, 54, 115], [503/885, 199/885, 157/885, 26/885]),
    'Recipient_Sex': {'0': 506/885, '1': 379/885},
    'Ethnicity': {'Hispanic': 128/885, 'Non-Hispanic': 757/885},
    'Race': {'White': 612/885, 'Black or African American': 145/885, 'Unknown': 128/885},
    'Disease_Type': {'AML': 356/885, 'ALL': 369/885, 'CML': 55/885, 'MDS': 105/885},
    'Conditioning_Regimen': {'Myeloablative': 632/885, 'Reduced intensity': 253/885},
    'Remission_Status': {'CR1': 260/885, 'CR2': 395/885, 'CR3': 0.0, 'Relapse': 230/885},
    'Cord_Blood_Units': {'Single': 1.0, 'Double': 0.0},
    'HLA_Match_Level': {'6/6': 114/885, '5/6': 317/885, '4/6': 351/885, '3/6': 88/885, 'Missing/Unknown': 15/885},
    'CD34_Dose': {'<=3.0x10^5/kg': 245/885, '>3.0x10^5/kg': 116/885, 'Missing/Unknown': 524/885},
    'TNC_Dose': {'<2.5x10^7/kg': 233/885, '>=2.5x10^7/kg': 603/885, 'Missing/Unknown': 49/885},
    'Neutrophil_Engraftment': {'1': 0.71, '0': 0.29},
    'Platelet_Engraftment': {'1': 0.57, '0': 0.43},
    'Chronic_GVHD': {'1': 0.24, '0': 0.76},
    '1_Year_Survival': {'1': 0.58, '0': 0.42}
}
dataset = add_study_data(study1_data, 'PMC3874400')
save_dataset(OUT, dataset)
validate_dataset(dataset)

# Study 2
study2_data = {
    'n_patients': 517,
    'Recipient_Age': ([0, 5, 16], [245/517, 272/517]),
    'Recipient_Sex': {'0': 280/517, '1': 237/517},
    'Ethnicity': {'Japanese': 257/517, 'Other/Unknown': 260/517},
    'Race': {'Asian': 257/517, 'White': 260/517},
    'Disease_Type': {'AML': 181/517, 'ALL': 336/517},
    'Conditioning_Regimen': {'Myeloablative': 360/517, 'Reduced intensity': 157/517},
    'Remission_Status': {'CR1': 249/517, 'CR2': 134/517, 'CR3': 134/517},
    'Cord_Blood_Units': {'Single': 1.0, 'Double': 0.0},
    'HLA_Match_Level': {'6/6': 106/517, '5/6': 249/517, '4/6': 162/517},
    'CD34_Dose': None,
    'TNC_Dose': {'<3.0x10^7/kg': 61/517, '>=3.0x10^7/kg': 448/517, 'Missing/Unknown': 8/517},
    'Neutrophil_Engraftment': {'1': 0.69, '0': 0.31},
    'Platelet_Engraftment': None,
    'Chronic_GVHD': {'1': 0.26, '0': 0.74},
    '1_Year_Survival': {'1': 0.59, '0': 0.41}
}
dataset = add_study_data(study2_data, 'S1083-8791')
save_dataset(OUT, dataset)
validate_dataset(dataset)

# Study 3
study3_data = {
    'n_patients': 193,
    'Recipient_Age': ([20, 30, 40, 50, 60, 70, 72], [11/193, 13/193, 31/193, 63/193, 70/193, 5/193]),
    'Recipient_Sex': {'0': 107/193, '1': 86/193},
    'Ethnicity': {'Hispanic': 20/193, 'Non-Hispanic': 162/193, 'Unknown': 11/193},
    'Race': {'White': 152/193, 'Black or African American': 19/193, 'Others': 19/193, 'Unknown': 3/193},
    'Disease_Type': {'AML': 108/193, 'ALL': 21/193, 'CLL': 8/193, 'CML': 3/193, 'Other acute leukemia': 3/193, 'Myelodysplastic disorders': 18/193, 'Non-Hodgkin lymphoma': 21/193, 'Hodgkin lymphoma': 11/193},
    'Conditioning_Regimen': {'Myeloablative': 1.0},
    'Remission_Status': None,
    'Cord_Blood_Units': {'Double': 1.0},
    'HLA_Match_Level': {'6/6': 7/193, '5/6': 59/193, '4/6': 119/193, '3/6': 7/193, 'Missing/Unknown': 1/193},
    'CD34_Dose': None,
    'TNC_Dose': {'median': 4.1, 'range': (1.1, 9.2), 'unit_exp': 7},
    'Neutrophil_Engraftment': {'1': 0.72, '0': 0.28},
    'Platelet_Engraftment': {'1': 0.74, '0': 0.26},
    'Chronic_GVHD': {'1': 0.30, '0': 0.70},
    '1_Year_Survival': {'1': 0.52, '0': 0.48}
}
dataset = add_study_data(study3_data, '7712')
save_dataset(OUT, dataset)
validate_dataset(dataset)

# Study 4
study4_data = {
    'n_patients': 190,
    'Recipient_Age': {'median': 58, 'range': (21, 73)},
    'Recipient_Sex': None,
    'Ethnicity': None,
    'Race': None,
    'Disease_Type': {'ALL': 29/190, 'AML': 92/190, 'CLL': 5/190, 'CML': 4/190, 'Hodgkin lymphoma': 1/190, 'MDS': 28/190, 'MDS/MPN': 9/190, 'MPN': 2/190, 'Non-Hodgkin lymphoma': 19/190, 'Other': 1/190},
    'Conditioning_Regimen': {'Myeloablative': 111/190, 'Reduced intensity': 8/190, 'Other': 71/190},
    'Remission_Status': None,
    'Cord_Blood_Units': {'Double': 1.0},
    'HLA_Match_Level': {'4/6,4/6': 26/190, '4/6,5/6': 39/190, '4/6,6/6': 2/190, '5/6,5/6': 78/190, '5/6,6/6': 23/190, '6/6,6/6': 22/190},
    'CD34_Dose': {'median': 0.21, 'range': (0.04, 0.70), 'unit_exp': 6},
    'TNC_Dose': {'median': 4.94, 'range': (2.84, 8.90), 'unit_exp': 7},
    'Neutrophil_Engraftment': {'1': 0.90, '0': 0.10},
    'Platelet_Engraftment': {'1': 0.84, '0': 0.16},
    'Chronic_GVHD': {'1': 0.18, '0': 0.82},
    '1_Year_Survival': {'1': 0.50, '0': 0.50}
}
dataset = add_study_data(study4_data, 'PMC7252552')
save_dataset(OUT, dataset)
validate_dataset(dataset)

# Study 5
study5_data = {
    'n_patients': 176,
    'Recipient_Age': {'median': 56, 'range': (18, 73)},
    'Recipient_Sex': {'0': 99/176, '1': 77/176},
    'Ethnicity': None,
    'Race': None,
    'Disease_Type': {'MDS': 1.0},
    'Conditioning_Regimen': {'Myeloablative': 61/176, 'Reduced intensity': 115/176},
    'Remission_Status': None,
    'Cord_Blood_Units': {'Single': 36/176, 'Double': 140/176},
    'HLA_Match_Level': {'3/6': 4/176, '4/6': 102/176, '5/6': 53/176, '6/6': 4/176, 'Missing/Unknown': 13/176},
    'CD34_Dose': {'0-2x10^5/kg': 83/176, '2-4x10^5/kg': 51/176, '4-8x10^5/kg': 16/176, '>8x10^5/kg': 14/176, 'Missing/Unknown': 12/176},
    'TNC_Dose': {'0-2x10^7/kg': 13/176, '2-4x10^7/kg': 74/176, '4-8x10^7/kg': 75/176, '>8x10^7/kg': 5/176, 'Missing/Unknown': 9/176},
    'Neutrophil_Engraftment': {'1': 0.92, '0': 0.08},
    'Platelet_Engraftment': {'1': 0.86, '0': 0.14},
    'Chronic_GVHD': {'1': 0.26, '0': 0.74},
    '1_Year_Survival': {'1': 0.47, '0': 0.53}
}
dataset = add_study_data(study5_data, 'PMC5474679')
save_dataset(OUT, dataset)
validate_dataset(dataset)

# Study 6
study6_data = {
    'n_patients': 1404,
    'Recipient_Age': {'range': (0, 79)},
    'Recipient_Sex': {'0': 748/1404, '1': 656/1404},
    'Ethnicity': {'Hispanic': 260/1404, 'Non-Hispanic': 1095/1404},
    'Race': {'White': 840/1404, 'Black or African American': 146/1404, 'Native American': 15/1404, 'Asian': 45/1404, 'Missing/Unknown': 49/1404},
    'Disease_Type': {'AML': 784/1404, 'ALL': 620/1404},
    'Conditioning_Regimen': {'Myeloablative': 1161/1404, 'Reduced intensity': 243/1404},
    'Remission_Status': {'CR1': 560/1404, 'CR2': 637/1404, 'Active': 205/1404, 'Missing/Unknown': 2/1404},
    'Cord_Blood_Units': {'Single': 810/1404, 'Double': 594/1404},
    'HLA_Match_Level': {'6/6': 169/1404, '5/6': 382/1404, '4/6': 259/1404, '4/6,4/6': 251/1404, '4/6,5/6': 125/1404, '5/6,5/6': 154/1404, '6/6,6/6': 64/1404},
    'CD34_Dose': None,
    'TNC_Dose': {
        '3-5x10^7/kg': 463/1404,
        '5-8x10^7/kg': 413/1404,
        '>=8x10^7/kg': 336/1404,
        'Missing/Unknown': 192/1404
    },
    'Neutrophil_Engraftment': None,
    'Platelet_Engraftment': None,
    'Chronic_GVHD': {'1': 0.27, '0': 0.73},
    '1_Year_Survival': None
}
dataset = add_study_data(study6_data, 'PMC5332289')
save_dataset(OUT, dataset)
validate_dataset(dataset)

# Study 7
study7_data = {
    'n_patients': 50,
    'Recipient_Age': {'median': 58, 'range': (16, 69)},
    'Recipient_Sex': {'0': 26/50, '1': 24/50},
    'Ethnicity': {'Hispanic or Latino': 2/50, 'Not Hispanic or Latino': 46/50, 'Unknown': 2/50},
    'Race': {'Asian': 4/50, 'Black or African American': 1/50, 'White': 45/50},
    'Disease_Type': {'ALL': 6/50, 'AML': 29/50, 'Biphenotypic/Undifferentiated Leukaemia': 1/50, 'Burkitts Lymphoma': 1/50, 'Hodgkins Lymphoma': 5/50, 'Large Cell Lymphoma': 3/50, 'Marginal Zone B-cell Lymphoma': 1/50, 'Follicular Non-Hodginks Lymphoma': 4/50},
    'Conditioning_Regimen': {'Reduced Intensity': 1.0},
    'Remission_Status': {'CR1': 23/50, 'CR2': 10/50, 'CR3': 3/50},
    'Cord_Blood_Units': {'Double': 1.0},
    'HLA_Match_Level': {'4/6,4/6': 21/50, '4/6,5/6': 11/50, '4/6,6/6': 1/50, '5/6,5/6': 13/50, '5/6,6/6': 1/50, '6/6,6/6': 3/50},
    'CD34_Dose': None,
    'TNC_Dose': {'median': 4.2, 'range': (2.3, 13.6), 'unit_exp': 7},
    'Neutrophil_Engraftment': {'1': 0.86, '0': 0.14},
    'Platelet_Engraftment': {'1': 0.82, '0': 0.18},
    'Chronic_GVHD': {'1': 0.25, '0': 0.75},
    '1_Year_Survival': {'1': 0.54, '0': 0.46}
}
dataset = add_study_data(study7_data, 'PMC3138683')
save_dataset(OUT, dataset)
validate_dataset(dataset)

# Study 8
study8_data = {
    'n_patients': 135,
    'Recipient_Age': ([16, 20, 30, 40, 53], [26/135, 37/135, 38/135, 34/135]),
    'Recipient_Sex': {'0': 78/135, '1': 56/135},
    'Ethnicity': None,
    'Race': None,
    'Disease_Type': {'AML': 74/135, 'ALL': 61/135},
    'Conditioning_Regimen': {'Myeloablative': 1.0},
    'Remission_Status': {'CR1': 73/135, 'CR2': 43/135, 'Relapse': 19/135},
    'Cord_Blood_Units': {'Single': 1.0, 'Double': 0.0},
    'HLA_Match_Level': {'6/6': 8/135, '5/6': 38/135, '4/6': 89/135},
    'CD34_Dose': {'median': 1.3, 'range': (0.1, 10), 'unit_exp': 5},
    'TNC_Dose': {'<1.5x10^7/kg': 8/135, '1.6-2.5x10^7/kg': 59/135, '2.6-3.5x10^7/kg': 35/135, '>3.5x10^7/kg': 24/135},
    'Neutrophil_Engraftment': {'1': 0.83, '0': 0.17},
    'Platelet_Engraftment': {'1': 0.70, '0': 0.30},
    'Chronic_GVHD': {'1': 0.43, '0': 0.57},
    '1_Year_Survival': None
}
dataset = add_study_data(study8_data, '1047/15724')
save_dataset(OUT, dataset)
validate_dataset(dataset)

# Study 9
n = 56
def _sample(med, lo, hi, n):
    sd = (hi - lo)/6 if hi > lo else 1.0
    v = np.random.normal(med, sd, n)
    return np.clip(v, lo, hi)
cd34_total = (_sample(1.09, 0.29, 7.06, n) + _sample(0.82, 0.11, 6.89, n)).tolist()
tnc_total = (_sample(2.62, 1.44, 5.62, n) + _sample(2.02, 1.07, 5.56, n)).tolist()
study9_data = {
    'n_patients': n,
    'Recipient_Age': {'median': 35, 'range': (18, 49)},
    'Recipient_Sex': None,
    'Ethnicity': {'Non-Hispanic': 34/56},
    'Race': {'White': 34/56},
    'Disease_Type': {'AML': 31/56, 'ALL': 19/56, 'Other Acute Leukaemia': 4/56, 'MDS': 2/56},
    'Conditioning_Regimen': {'Myeloablative': 1.0},
    'Remission_Status': {'CR1': 28/56, 'CR2': 26/56},
    'Cord_Blood_Units': {'Double': 1.0},
    'HLA_Match_Level': {'6/6': 4/56, '5/6': 40/56, '4/6': 12/56},
    'CD34_Dose': cd34_total, # defaults to 10^5/kg
    'TNC_Dose': tnc_total,   # defaults to 10^7/kg
    'Neutrophil_Engraftment': {'1': 0.78, '0': 0.22},
    'Platelet_Engraftment': {'1': 0.65, '0': 0.35},
    'Chronic_GVHD': {'1': 0.36, '0': 0.64},
    '1_Year_Survival': {'1': 0.57, '0': 0.43}
}
dataset = add_study_data(study9_data, 'PMC5557396')
save_dataset(OUT, dataset)
validate_dataset(dataset)

# Study 10
study10_data = {
    'n_patients': 224,
    'Recipient_Age': {'median': 10.15, 'range': (1.1, 21.4)},
    'Recipient_Sex': {'0': 128/224, '1': 96/224},
    'Ethnicity': {'Hispanic or Latino': 42/224, 'Not Hispanic or Latino': 176/224, 'Unknown/Missing': 6/224},
    'Race': {'American Indian or Alaskan Native': 1/224, 'Asian': 9/224, 'Black': 24/224, 'White': 165/224, 'More than one race': 5/224, 'Other': 4/224, 'Unknown/Missing': 15/224},
    'Disease_Type': {'AML': 77/224, 'ALL': 119/224, 'Acute Biphenotypic Leukaemia': 8/224, 'Acute Undifferentiated Leukaemia': 1/224, 'Chronic Myeloid Leukaemia': 1/224, 'Myelodysplastic Syndrome': 18/224},
    'Conditioning_Regimen': {'Myeloablative': 1.0},
    'Remission_Status': {'CR1': 75/224, 'CR2': 94/224, 'Relapse': 4/224},
    'Cord_Blood_Units': {'Double': 111/224, 'Single': 113/224},
    'HLA_Match_Level': {'3/6': 3/224, '4/6': 92/224, '5/6': 97/224, '6/6': 28/224},
    'CD34_Dose': {'median': 2.8, 'range': (0.1, 10.0), 'unit_exp': 5},
    'TNC_Dose': {'median': 5.55, 'range': (1.0, 12.0), 'unit_exp': 7},
    'Neutrophil_Engraftment': {'1': 0.89, '0': 0.11},
    'Platelet_Engraftment': {'1': 0.78, '0': 0.22},
    'Chronic_GVHD': {'1': 0.31, '0': 0.69},
    '1_Year_Survival': {'1': 0.69, '0': 0.31}
}
dataset = add_study_data(study10_data, 'NEJMoa1405584')
save_dataset(OUT, dataset)
validate_dataset(dataset)

# Study 11
study11_data = {
    'n_patients': 333,
    'Recipient_Age': ([0, 16, 29, 39, 49], [154/333, 87/333, 55/333, 37/333]),
    'Recipient_Sex': {'0': 192/333, '1': 141/333},
    'Ethnicity': None,
    'Race': {'Caucasian': 213/333, 'Non-Caucasian': 86/333, 'Unknown': 34/333},
    'Disease_Type': {'AML': 125/333, 'ALL': 208/333},
    'Conditioning_Regimen': {'Myeloablative': 1.0},
    'Remission_Status': {'CR1': 166/333, 'CR2': 167/333},
    'Cord_Blood_Units': {'Single': 69/333, 'Double': 113/333},
    'HLA_Match_Level': {'<=5/8': 188/333, '6-8/8': 145/333},
    'CD34_Dose': None,
    'TNC_Dose': None,
    'Neutrophil_Engraftment': {'1': 0.77, '0': 0.23},
    'Platelet_Engraftment': {'1': 0.80, '0': 0.20},
    'Chronic_GVHD': {'1': 0.41, '0': 0.59},
    '1_Year_Survival': {'1': 0.70, '0': 0.30}
}
dataset = add_study_data(study11_data, '4064/476719')
save_dataset(OUT, dataset)
validate_dataset(dataset)

# Study 12
study12_data = {
    'n_patients': 118,
    'Recipient_Age': ([18, 30, 40, 50, 60, 70], [27/118, 35/118, 29/118, 14/118, 13/118]),
    'Recipient_Sex': {'0': 58/118, '1': 60/118},
    'Ethnicity': None,
    'Race': {'African American': 1.0},
    'Disease_Type': {'AML': 56/118, 'ALL': 26/118, 'Myelodysplastic syndrome': 13/118, 'Non-Hodgkin lymphoma': 23/118},
    'Conditioning_Regimen': {'Myeloablative': 85/118, 'Reduced intensity': 33/118},
    'Remission_Status': {'CR1': 50/118, 'CR2': 32/118},
    'Cord_Blood_Units': {'Double': 103/118, 'Single': 15/118},
    'HLA_Match_Level': {'4/6': 106/118, '>=5/6': 12/118},
    'CD34_Dose': None,
    'TNC_Dose': {'median': 4.25, 'range': (3.9,7.2), 'unit_exp': 7},
    'Neutrophil_Engraftment': {'1': 0.75, '0': 0.25},
    'Platelet_Engraftment': {'1': 0.71, '0': 0.29},
    'Chronic_GVHD': {'1': 0.26, '0': 0.74},
    '1_Year_Survival': {'1': 0.56, '0': 0.44}
}
dataset = add_study_data(study12_data, 'PMC7530013')
save_dataset(OUT, dataset)
validate_dataset(dataset)

# Study 13
study13_data = {
    'n_patients': 102,
    'Recipient_Age': {'median':7.4, 'range':(0.2,56.9)},
    'Recipient_Sex': {'0': 60/102, '1': 42/102},
    'Ethnicity': None,
    'Race': {'White': 80/102, 'Other': 22/102},
    'Disease_Type': {'ALL':28/102, 'AML':26/102, 'Chronic Myelogenous Leukaemia':6/102, 'Juvenile Myelomonocytic Leukaemia':3/102, 'Non-Hodgkin lymphoma':1/102, 'Hodgkin disease':1/102, 'Severe aplastic anaemia':2/102, 'Fanconi anaemia':4/102, 'Diamond-Blackfan syndrome':1/102, 'Osteopetrosis':1/102, 'Myelodysplastic syndrome':3/102, 'Immune deficiency':5/102, 'Metabolic disorders':21/102},
    'Conditioning_Regimen': {'Myeloablative': 1.0},
    'Remission_Status': {'CR1':9/102, 'CR2':24/102, 'CR3':7/102, 'Relapse':14/102},
    'Cord_Blood_Units': {'Single': 1.0},
    'HLA_Match_Level': {'6/6':14/102, '5/6':44/102, '4/6':42/102, '3/6':2/102},
    'CD34_Dose': {'median':2.8, 'range':(0.4,39.1), 'unit_exp': 5},
    'TNC_Dose': {'median':3.1, 'range':(0.7,57.9), 'unit_exp': 7},
    'Neutrophil_Engraftment': {'1': 0.84, '0': 0.16},
    'Platelet_Engraftment': {'1': 0.57, '0': 0.43},
    'Chronic_GVHD': {'1': 0.09, '0': 0.91},
    '1_Year_Survival': {'1': 0.58, '0': 0.42}
}
dataset = add_study_data(study13_data, '1611/106372')
save_dataset(OUT, dataset)
validate_dataset(dataset)

# Study 14
study14_data = {
    'n_patients': 205,
    'Recipient_Age': {'median':59, 'range':(50,71)},
    'Recipient_Sex': {'0': 99/205, '1': 105/205},
    'Ethnicity': None,
    'Race': None,
    'Disease_Type': {'AML': 1.0},
    'Conditioning_Regimen': {'Myeloablative': 42/205, 'Reduced intensity': 137/205},
    'Remission_Status': {'CR1': 1.0},
    'Cord_Blood_Units': {'Single': 80/205, 'Double': 125/205},
    'HLA_Match_Level': {'6/6': 8/205, '4/6 or 5/6': 197/205},
    'CD34_Dose': None,
    'TNC_Dose': None,
    'Neutrophil_Engraftment': {'1': 0.69, '0': 0.31},
    'Platelet_Engraftment': {'1': 0.71, '0': 0.29},
    'Chronic_GVHD': {'1': 0.28, '0': 0.72},
    '1_Year_Survival': {'1': 0.58, '0': 0.42}
}
dataset = add_study_data(study14_data, 'PMC4085692')
save_dataset(OUT, dataset)
validate_dataset(dataset)

# Study 15
study15_data = {
    'n_patients': 183,
    'Recipient_Age': {'median':10, 'range':(0.42,20)},
    'Recipient_Sex': None,
    'Ethnicity': None,
    'Race': None,
    'Disease_Type': {'AML': 1.0},
    'Conditioning_Regimen': {'Myeloablative': 1.0},
    'Remission_Status': {'CR1': 80/183, 'CR2': 103/183},
    'Cord_Blood_Units': {'Single': 122/183, 'Double': 61/183},
    'HLA_Match_Level': {'6/6': 46/183, '5/6': 83/183, '<=4/6': 20/183},
    'CD34_Dose': {'median':0.32, 'range':(0.02,3), 'unit_exp': 5},
    'TNC_Dose': {'median':0.545, 'range':(0.1,32), 'unit_exp': 7},
    'Neutrophil_Engraftment': {'1': 0.89, '0': 0.11},
    'Platelet_Engraftment': {'1': 0.74, '0': 0.26},
    'Chronic_GVHD': {'1': 0.21, '0': 0.79},
    '1_Year_Survival': {'1': 0.62, '0': 0.38}
}
dataset = add_study_data(study15_data, '1118/247275')
save_dataset(OUT, dataset)
validate_dataset(dataset)

# Study 16
study16_data = {
    'n_patients': 2512,
    'Recipient_Age': ([1, 9, 17, 19, 29, 39, 49, 59, 69, 79], [522/2512, 285/2512, 66/2512, 255/2512, 271/2512, 275/2512, 371/2512, 408/2512, 59/2512]),
    'Recipient_Sex': {'0': 1321/2512, '1': 1191/2512},
    'Ethnicity': {'Hispanic/Latinx': 496/2512, 'Non-Hispanic': 2016/2512},
    'Race': {'Caucasian(white)': 1517/2512, 'African-American(Black)': 322/2512, 'Asian': 177/2512, 'Unknown': 496/2512},
    'Disease_Type': {'AML': 1418/2512, 'ALL': 816/2512, 'Myelodysplastic syndrome': 278/2512},
    'Conditioning_Regimen': {'Myeloablative': 1752/2512, 'Reduced intensity': 759/2512, 'Missing': 1/2512},
    'Remission_Status': {'CR1': 1075/2512, 'CR2': 850/2512, 'CR3+': 309/2512},
    'Cord_Blood_Units': {'Single': 983/2512, 'Double': 1529/2512},
    'HLA_Match_Level': {'4/6': 1158/2512, '5/6': 861/2512, '6/6': 199/2512, 'Missing/Unknown': 294/2512},
    'CD34_Dose': {'median': 3.25, 'range': (0, 10), 'unit_exp': 5},
    'TNC_Dose': {'median': 4.75, 'range': (0, 10), 'unit_exp': 7},
    'Neutrophil_Engraftment': {'1': 0.71, '0': 0.29},
    'Platelet_Engraftment': {'1': 0.72, '0': 0.28},
    'Chronic_GVHD': {'1': 0.27, '0': 0.73},
    '1_Year_Survival': {'1': 0.57, '0': 0.43}
}
dataset = add_study_data(study16_data, 'S2666-6367')
save_dataset(OUT, dataset)
validate_dataset(dataset)

# Study 17
study17_data = {
    'n_patients': 186,
    'Recipient_Age': {'median':58, 'range':(20,70)},
    'Recipient_Sex': {'0': 97/186, '1': 89/186},
    'Ethnicity': {'Hispanic': 22/186, 'Non-Hispanic': 164/186},
    'Race': {'White': 145/186, 'African American': 27/186, 'Other': 14/186},
    'Disease_Type': {'ALL':31/186, 'AML':98/186, 'Biphenotypic leukaemia':1/186, 'T-cell leukaemia/lymphoma':4/186, 'Hodgkin lymphoma':10/186, 'Large cell lymphoma':21/186, 'Follicular non-hodgkin lymphoma':7/186, 'Mantle cell lymphoma':6/186, 'Other lymphoma':7/186},
    'Conditioning_Regimen': {'Myeloablative': 1.0},
    'Remission_Status': {'CR1':99/186, 'CR2':35/186},
    'Cord_Blood_Units': {'Double': 1.0},
    'HLA_Match_Level': {'>=4/6': 1.0},
    'CD34_Dose': {'median':0.13, 'range':(0.13,0.13), 'unit_exp': 5},
    'TNC_Dose': {'median':2.95, 'range':(2.95,2.95), 'unit_exp': 7},
    'Neutrophil_Engraftment': {'1': 0.90, '0': 0.10},
    'Platelet_Engraftment': {'1': 0.78, '0': 0.22},
    'Chronic_GVHD': {'1': 0.22, '0': 0.78},
    '1_Year_Survival': {'1': 0.65, '0': 0.35}
}
dataset = add_study_data(study17_data, 'PMC7819761')
save_dataset(OUT, dataset)
validate_dataset(dataset)

# Study 18
study18_data = {
    'n_patients': 449,
    'Recipient_Age': ([0, 20, 40, 80], [84/449, 118/449, 247/449]),
    'Recipient_Sex': None,
    'Ethnicity': None,
    'Race': None,
    'Disease_Type': {'AML':296/449, 'ALL':153/449},
    'Conditioning_Regimen': {'Reduced intensity':213/449, 'Myeloablative':220/449},
    'Remission_Status': {'CR1':224/449, 'CR2':157/449, 'CR3':37/449},
    'Cord_Blood_Units': {'Double': 1.0},
    'HLA_Match_Level': {'4/6,4/6':164/449, '4/6,5/6':107/449, '4/6,6/6':2/449, '5/6,5/6':120/449, '5/6,6/6':26/449, '6/6,6/6':30/449},
    'CD34_Dose': None,
    'TNC_Dose': {'median':4.60, 'range':(2.31,6.00), 'unit_exp': 7},
    'Neutrophil_Engraftment': {'1': 0.68, '0': 0.32},
    'Platelet_Engraftment': {'1': 0.72, '0': 0.28},
    'Chronic_GVHD': {'1': 0.29, '0': 0.71},
    '1_Year_Survival': {'1': 0.46, '0': 0.54}
}
dataset = add_study_data(study18_data, 'PMC5477613')
save_dataset(OUT, dataset)
validate_dataset(dataset)

# Study 19
study19_data = {
    'n_patients': 1742,
    'Recipient_Age': {'median':7, 'range':(0.3,17.9)},
    'Recipient_Sex': {'0': 737/1742, '1': 997/1742},
    'Ethnicity': None,
    'Race': None,
    'Disease_Type': {'AML':708/1742, 'ALL':1034/1742},
    'Conditioning_Regimen': {'Myeloablative':1531/1742, 'Reduced intensity':135/1742},
    'Remission_Status': {'CR1': 723/1742, 'CR2': 702/1742, 'CR3': 94/1742, 'Active': 137/1742},
    'Cord_Blood_Units': {'Single': 1.0},
    'HLA_Match_Level': {'6/6': 320/1742, '5/6': 848/1742, '4/6': 392/1742, '3/6': 23/1742, '2/6': 3/1742},
    'CD34_Dose': {'median':2.0, 'range':(0.1,32.6), 'unit_exp': 5},
    'TNC_Dose': {'median':8.0, 'range':(0.5,44.4), 'unit_exp': 7},
    'Neutrophil_Engraftment': {'1': 0.88, '0': 0.12},
    'Platelet_Engraftment': None,
    'Chronic_GVHD': {'1': 0.12, '0': 0.88},
    '1_Year_Survival': {'1': 0.56, '0': 0.44}
}
dataset = add_study_data(study19_data, 'S2666636724004895')
save_dataset(OUT, dataset)
validate_dataset(dataset)

# Study 20
study20_data = {
    'n_patients': 562,
    'Recipient_Age': ([0, 2, 5, 11, 17, 80], [114/562, 127/562, 137/562, 82/562, 102/562]),
    'Recipient_Sex': {'0': 324/562, '1': 238/562},
    'Ethnicity': None,
    'Race': {'White': 405/562, 'Non-White': 157/562},
    'Disease_Type': {'ALL':177/562, 'AML':124/562, 'CML':48/562, 'JCML':14/562, 'CLL':2/562, 'Lymphoma':13/562, "Fanconi's anaemia":35/562, 'SCID':24/562, 'Osteopetrosis':12/562, "Hurler's syndrome":8/562, 'Wiskott-Aldrich syndrome':7/562, 'Adrenoleukodystrophy':6/562, 'Blackfan-Diamond Syndrome':5/562, 'Other genetic disease':40/562, 'Myelodysplastic disease':21/562, 'Severe aplastic anaemia':21/562, 'Other cancer':5/562},
    'Conditioning_Regimen': None,
    'Remission_Status': {'CR1':56/562, 'CR2':166/562, 'CR3':116/562},
    'Cord_Blood_Units': {'Single': 1.0},
    'HLA_Match_Level': {'6/6': 40/562, '5/6': 218/562, '4/6': 261/562, '3/6': 37/562, '2/6': 3/562},
    'CD34_Dose': None,
    'TNC_Dose': None,
    'Neutrophil_Engraftment': {'1': 0.62, '0': 0.38},
    'Platelet_Engraftment': {'1': 0.57, '0': 0.43},
    'Chronic_GVHD': {'1': 0.09, '0': 0.91},
    '1_Year_Survival': {'1': 0.42, '0': 0.58}
}
dataset = add_study_data(study20_data, 'NEJM199811263392201')
save_dataset(OUT, dataset)
validate_dataset(dataset)

print("Final shape:", dataset.shape)

# Quick sanity test for the CD34 inequality sampling 
print("\nSanity check on Study 1 CD34 (should NOT be a point mass at 3.0):")
s1 = dataset.loc[dataset['Study_ID']=='PMC3874400','CD34_num']
print(s1.describe())
print("Unique near 3:", np.sum(np.isclose(s1, 3.0, atol=1e-6)), "of", s1.notna().sum())


Python executable: /opt/anaconda3/bin/python
NumPy version: 1.26.4
Pandas version: 2.2.2
Added 885 patients from PMC3874400
Dataset saved at: /Users/amanda/Desktop/UCBT/ucbt_dataset.csv

VALIDATION 
Top missing columns:
hla_any6_pre                0.010169
hla_worst_pre               0.010169
hla_best_pre                0.010169
Patient_ID                  0.000000
Recipient_Age               0.000000
CD34_num                    0.000000
hla_double_pre              0.000000
TNC_num_pre                 0.000000
CD34_num_pre                0.000000
TNC_Dose_missing            0.000000
CD34_Dose_missing           0.000000
Remission_Status_missing    0.000000
dtype: float64

Median CD34_num by Study_ID (expected 10^5/kg scale):
Study_ID
PMC3874400    2.522997
Name: CD34_num, dtype: float64
...
Study_ID
PMC3874400    2.522997
Name: CD34_num, dtype: float64

Median TNC_num by Study_ID (expected 10^7/kg scale):
Study_ID
PMC3874400    2.537983
Name: TNC_num, dtype: float64
...
Study_ID
PMC3874